# The table everyone thinks they agree about

Four people describe the same dataset. One says it is 48 rows. One says it is four plots
over twelve weeks. One says the third column is spend. One says the third column is spend
*in thousands*. The model gets fit on whatever the loader returned, and the disagreement
surfaces at the review, in the form of a number that is a thousand times too big.

A `Panel` is a units × time table plus a `RoleMap` that says which column is the unit index,
which is time, and which entity each measured column is. **A column adopts the dimension of
its role** (review D3): units are declared once, here, and nothing else in an analysis
mentions them.

And the panel validates and *reports*; it never imputes. A loader that fills gaps is a
loader that decides — quietly, before anyone has stated an assumption — what the missing
weeks were worth.

In [ ]:
import numpy as np
import pandas as pd

from axiom.core import Covariate, D, Outcome, Treatment, dimensionless
from axiom.data import Completeness, Panel, PanelError, RoleKind, RoleMap

from axiom.display import enable, table

import sys; sys.path[:0] = ["..", "../.."]  # nbs/ is on the path either way
from _style import caption, heat

enable();  # every axiom result renders itself from here on

In [ ]:
rng = np.random.default_rng(7)
units, periods = ["p01", "p02", "p03", "p04"], range(12)
df = pd.DataFrame(
    {
        "plot": np.repeat(units, len(periods)),
        "week": np.tile(list(periods), len(units)),
        "yield_kg": rng.gamma(5.0, 40.0, 48),
        "fert_usd": rng.uniform(0, 200, 48),
        "irrig_usd": rng.uniform(0, 80, 48),
        "rain_z": rng.normal(0, 1, 48),
    }
).sample(frac=1, random_state=1)  # shuffled on purpose; Panel sorts
df.head()

In [ ]:
roles = RoleMap(
    unit="plot",
    time="week",
    outcome=("yield_kg", Outcome(name="yield_total", dimension=D.outcome, unit="kg")),
    treatments={
        "fert_usd": Treatment(name="fertilizer", dimension=D.currency, unit="USD"),
        "irrig_usd": Treatment(name="irrigation", dimension=D.currency, unit="USD"),
    },
    covariates={"rain_z": Covariate(name="rainfall", dimension=dimensionless())},
)
print(roles.columns)
print(roles.measured)
kind: RoleKind = roles.kind_of("fert_usd")
print(kind, roles.dimension_of("fert_usd"), roles.unit_of("yield_kg"))

The `RoleMap` is the disagreement resolved in one object: `fert_usd` is a `Treatment` named
fertilizer, measured in currency, in USD. Nothing downstream has to be told again, and
nothing downstream is *able* to be told differently.

In [ ]:
panel = Panel(df, roles)
panel

In [ ]:
print(panel.units)
print(panel.periods[:5], "...")
print(panel.frame.head(3))

## Shapes for estimators

`column` gives the long vector; `array` gives `(n_units, n_periods)` with NaN where a cell
is absent; `wide` is the labelled version.

In [ ]:
print(panel.column("yield_kg").shape, panel.array("fert_usd").shape)
panel.wide("irrig_usd").iloc[:, :4]

In [ ]:
fig = heat(
    panel.array("fert_usd"), [f"w{p}" for p in panel.periods], list(panel.units),
    text_fmt="{:.0f}",
    colorbar_title="USD",
    title="What the estimator is actually handed",
    subtitle="fertilizer dose, four plots × twelve weeks — the (n_units, n_periods) array",
    x_title="week",
    height=260,
)
caption(fig, "The rows are sorted and the columns are the declared periods, whatever order "
             "the file arrived in — this panel was shuffled on construction.")

## Completeness is reported, not fixed (review D5)

In [ ]:
from axiom.display import show

c: Completeness = panel.completeness()
show(c)

In [ ]:
holey = Panel(df.iloc[3:], roles)            # drop three rows somewhere
c = holey.completeness()
print(c.balanced, c.missing_cells, c.gaps)
print("NaNs in the wide array:", int(np.isnan(holey.array("yield_kg")).sum()))

try:
    holey.require_balanced(context="a synthetic-control estimator")
except PanelError as e:
    print("refused:", e)

In [ ]:
fig = heat(
    holey.array("yield_kg"), [f"w{p}" for p in holey.periods], list(holey.units),
    text_fmt="{:.0f}",
    colorbar_title="kg",
    title="Missingness has a shape",
    subtitle=f"{c.missing_cells} absent cells — a count would say how many, not which",
    x_title="week",
    height=260,
)
caption(fig, "Holes concentrated in one unit's early weeks and holes scattered at random are "
             "different problems with different fixes, and only one of them is safe to "
             "hand to an estimator that assumes a balanced panel.")

## Validation failures are loud

Each of these is a data-loading mistake that would otherwise reach the model as a column of
`NaN`, a silently ignored feature, or a string coerced to a category.

In [ ]:
refused = []
for label, bad in (
    ("a column the roles name is missing", lambda: Panel(df.drop(columns=["rain_z"]), roles)),
    ("a column the roles do not name", lambda: Panel(df.assign(extra=1.0), roles)),
    ("a numeric column that is not numeric", lambda: Panel(df.assign(yield_kg="n/a"), roles)),
):
    try:
        bad()
    except PanelError as e:
        refused.append([label, str(e)])
table(refused, headers=("panel", "PanelError"))

## Identity

`content_hash()` covers the roles and the data, is row-order invariant, and is what `io`
records in provenance — so "we re-ran it on the same data" is checkable, including when the
export happened to come out in a different order.

In [ ]:
print(panel.content_hash()[:16] == Panel(df.sample(frac=1, random_state=3), roles).content_hash()[:16])
print(panel.select_units(["p02", "p04"]).units)

## What this bought you

One object that carries the data, what each column *is*, and what is missing from it —
validated at construction, hashed for provenance, and never repaired behind your back.

`02-scaling.ipynb` is the other half: fitting on scaled columns without losing the units
this notebook just pinned down.